# 한국어 영어 번역기 과제

## Step 1. 데이터 다운로드

In [12]:
# ─────────────────────────────────────────────────────────────
# [Cell 1] 환경 설정 및 데이터 다운로드
# 필요한 패키지를 설치하고, GitHub에서 한-영 병렬 코퍼스를 다운로드합니다.
# ─────────────────────────────────────────────────────────────

# ── 패키지 설치 ──────────────────────────────────────────────
!pip install konlpy -q

!apt-get install -y fonts-nanum > /dev/null 2>&1

import os
import re
import urllib.request

import tarfile
import logging
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from konlpy.tag import Okt


# ── 디바이스 설정 ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")
print(f"PyTorch 버전: {torch.__version__}")

# ── OKT 형태소 분석기 초기화 ──────────────────────────────────
okt = Okt()
print("OKT 초기화 완료")

url = "https://raw.githubusercontent.com/jungyeul/korean-parallel-corpora/master/korean-english-news-v1/korean-english-park.train.tar.gz"
tar_filename = "korean-english-park.train.tar.gz"
extract_path = "./data"  # 압축을 풀 폴더 경로
# 2. data 폴더가 없다면 생성
os.makedirs(extract_path, exist_ok=True)

사용 디바이스: cuda
PyTorch 버전: 2.7.1+cu118
OKT 초기화 완료


In [13]:
# 3. 데이터 다운로드
print("데이터 다운로드 중...")
try:
    urllib.request.urlretrieve(url, tar_filename)
    print(f"다운로드 완료: {tar_filename}")
    
    # 4. tar.gz 압축 해제
    print("압축 해제 중...")
    with tarfile.open(tar_filename, "r:gz") as tar:
        tar.extractall(path=extract_path)
    print(f"압축 해제 완료! 파일이 '{extract_path}' 폴더에 저장되었습니다.")

except Exception as e:
    print(f"오류가 발생했습니다: {e}")

데이터 다운로드 중...
다운로드 완료: korean-english-park.train.tar.gz
압축 해제 중...


/tmp/ipykernel_127/3120772059.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


압축 해제 완료! 파일이 './data' 폴더에 저장되었습니다.


In [14]:
# 5. 다운로드한 원본 tar.gz 파일 삭제
if os.path.exists(tar_filename):
    os.remove(tar_filename)
    print(f"임시 파일 삭제 완료: {tar_filename}")

# 6. 추출된 파일 확인
print("\n[ 추출된 파일 목록 ]")
print(os.listdir(extract_path))

임시 파일 삭제 완료: korean-english-park.train.tar.gz

[ 추출된 파일 목록 ]
['.ipynb_checkpoints', 'korean-english-park.train.en', 'korean-english-park.train.ko']


## Step 2. 데이터 정제

In [15]:
# ─────────────────────────────────────────────────────────────
# [Cell 2] 데이터 로드 및 정제
# - 압축 해제된 파일 경로 확인 후 로드
# - set을 활용한 중복 병렬 쌍 제거
# - 한글/영어 각각 정규식 전처리
# - OKT 형태소 분석 + 토큰 길이 40 이하 필터링
# ─────────────────────────────────────────────────────────────

import os
import re
from tqdm import tqdm

# ── 압축 해제된 파일 경로 확인 ────────────────────────────────
# tar.gz 해제 후 실제 파일명/경로를 확인합니다
extract_path = "./data"
print("[ 추출된 파일 목록 ]")
for root, dirs, files in os.walk(extract_path):
    for f in files:
        print(os.path.join(root, f))

# ── 파일 경로 설정 ────────────────────────────────────────────
# os.walk 결과를 보고 실제 경로에 맞게 아래 경로를 확인하세요
ko_path = "./data/korean-english-park.train.ko"
en_path = "./data/korean-english-park.train.en"

# ── 원시 데이터 로드 ──────────────────────────────────────────
with open(ko_path, "r", encoding="utf-8") as f:
    ko_lines = f.read().splitlines()

with open(en_path, "r", encoding="utf-8") as f:
    en_lines = f.read().splitlines()

print(f"\n원본 데이터 수: {len(ko_lines):,}")
print(f"[한국어 샘플] {ko_lines[0]}")
print(f"[영어   샘플] {en_lines[0]}")

# ── 전처리 함수 정의 ──────────────────────────────────────────
def preprocess_english(sentence: str) -> str:
    """
    영어 문장 정규화
    - 소문자 변환
    - 구두점 앞뒤 공백 삽입 (토큰 분리 목적)
    - 알파벳·구두점 외 문자 제거
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)    # 구두점 분리
    sentence = re.sub(r'[" "]+', " ", sentence)           # 연속 공백 제거
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)  # 허용 문자 외 제거
    return sentence.strip()

def preprocess_korean(sentence: str) -> str:
    """
    한국어 문장 정규화
    - 한글·숫자·공백 외 문자 제거
    - 연속 공백 정리
    """
    sentence = sentence.strip()
    sentence = re.sub(r"[^ 가-힣0-9]", " ", sentence)    # 한글·숫자·공백만 유지
    sentence = re.sub(r"\s+", " ", sentence)              # 연속 공백 제거
    return sentence.strip()

# ── 중복 제거 ─────────────────────────────────────────────────
# (한글, 영어) 튜플을 set에 추가 → 동일 쌍 자동 제거
# 병렬 쌍이 흐트러지지 않도록 반드시 튜플로 묶어서 처리
seen = set()
cleaned_corpus = []  # [(ko_clean, en_clean), ...]

for ko, en in zip(ko_lines, en_lines):
    ko_clean = preprocess_korean(ko)
    en_clean = preprocess_english(en)
    pair = (ko_clean, en_clean)
    # 빈 문장이거나 이미 등장한 쌍이면 건너뜀
    if pair not in seen and ko_clean and en_clean:
        seen.add(pair)
        cleaned_corpus.append(pair)

print(f"중복 제거 후 데이터 수: {len(cleaned_corpus):,}")

# ── OKT 형태소 분석 + 길이 필터링 ────────────────────────────
# norm=True : 정규화 (ㅋㅋ→크크 등)
# stem=True : 어간 추출 (먹었다→먹다)
MAX_TOKEN_LEN = 40

kor_corpus = []  # 한국어 형태소 토큰 리스트
eng_corpus = []  # 영어 토큰 리스트 (<start>/<end> 포함)

print("\nOKT 형태소 분석 중... (시간이 걸릴 수 있습니다)")
for ko_clean, en_clean in tqdm(cleaned_corpus):
    ko_tokens = okt.morphs(ko_clean, norm=True, stem=True)
    en_tokens = en_clean.split()

    # 양쪽 모두 길이 40 이하인 쌍만 사용
    if len(ko_tokens) <= MAX_TOKEN_LEN and len(en_tokens) <= MAX_TOKEN_LEN:
        kor_corpus.append(ko_tokens)
        eng_corpus.append(["<start>"] + en_tokens + ["<end>"])  # 영어에 시작/종료 토큰 추가

print(f"필터링 후 최종 데이터 수: {len(kor_corpus):,}")
print(f"\n[샘플 확인]")
print(f"한국어: {kor_corpus[0]}")
print(f"영  어: {eng_corpus[0]}")

[ 추출된 파일 목록 ]
./data/korean-english-park.train.en
./data/korean-english-park.train.ko
./data/.ipynb_checkpoints/korean-english-park.train-checkpoint.en

원본 데이터 수: 94,123
[한국어 샘플] 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
[영어   샘플] Much of personal computing is about "can you top this?"
중복 제거 후 데이터 수: 78,884

OKT 형태소 분석 중... (시간이 걸릴 수 있습니다)


100%|██████████| 78884/78884 [09:33<00:00, 137.52it/s]

필터링 후 최종 데이터 수: 69,343

[샘플 확인]
한국어: ['개인', '용', '컴퓨터', '사용', '의', '상당', '부분', '은', '이', '것', '보다', '뛰어나다', '수', '있다']
영  어: ['<start>', 'much', 'of', 'personal', 'computing', 'is', 'about', 'can', 'you', 'top', 'this', '?', '<end>']


## Step 3. 데이터 토큰화

In [16]:
# ─────────────────────────────────────────────────────────────
# [Cell 3] 단어 사전 구축 및 데이터셋 구성
# - 한국어/영어 vocab 구축 (min_freq 기반 희귀 단어 제거)
# - 정수 인코딩
# - PyTorch Dataset / DataLoader 생성
# ─────────────────────────────────────────────────────────────

from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader

# ── 특수 토큰 ID 정의 ─────────────────────────────────────────
PAD_ID = 0   # 패딩 (시퀀스 길이 통일)
BOS_ID = 1   # 문장 시작 <start>
EOS_ID = 2   # 문장 끝   <end>
UNK_ID = 3   # 미등록 단어 <unk>

SPECIAL_TOKENS = ["<pad>", "<start>", "<end>", "<unk>"]

# ── 단어 사전 구축 함수 ───────────────────────────────────────
def build_vocab(corpus: list, min_freq: int = 2) -> dict:
    """
    corpus  : 토큰 리스트의 리스트
    min_freq: 최소 등장 횟수 (희귀 단어 제거)
    반환    : {token: id} 딕셔너리
    """
    counter = Counter(token for sentence in corpus for token in sentence)
    vocab   = {tok: idx for idx, tok in enumerate(SPECIAL_TOKENS)}
    for word, freq in counter.most_common():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

# ── vocab 구축 ────────────────────────────────────────────────
kor_vocab = build_vocab(kor_corpus, min_freq=2)
eng_vocab = build_vocab(eng_corpus, min_freq=2)

# 디코딩용 역방향 매핑 (id → token)
id2kor = {v: k for k, v in kor_vocab.items()}
id2eng = {v: k for k, v in eng_vocab.items()}

print(f"한국어 vocab 크기: {len(kor_vocab):,}")
print(f"영어   vocab 크기: {len(eng_vocab):,}")

한국어 vocab 크기: 24,270
영어   vocab 크기: 26,348


In [17]:
# ── 정수 인코딩 ───────────────────────────────────────────────
def encode(tokens: list, vocab: dict) -> list:
    """토큰 리스트 → ID 리스트 (미등록 단어는 UNK_ID)"""
    return [vocab.get(t, UNK_ID) for t in tokens]

kor_encoded = [encode(s, kor_vocab) for s in kor_corpus]
eng_encoded = [encode(s, eng_vocab) for s in eng_corpus]

In [18]:
# ── PyTorch Dataset ───────────────────────────────────────────
class TranslationDataset(Dataset):
    """
    한국어(src) → 영어(trg) 번역 데이터셋
    - src      : 한국어 형태소 ID 시퀀스 (패딩 포함)
    - trg_input: <start> ~ EOS 직전 (디코더 입력)
    - trg_label: BOS 이후 ~ <end>   (손실 계산 정답)
    """
    def __init__(self, kor_data: list, eng_data: list, max_len: int):
        self.kor_data = kor_data
        self.eng_data = eng_data
        self.max_len  = max_len

    def __len__(self):
        return len(self.kor_data)

    def __getitem__(self, idx):
        src_ids = self.kor_data[idx][:self.max_len]
        trg_ids = self.eng_data[idx]  # 이미 <start>/<end> 포함

        trg_input = trg_ids[:-1][:self.max_len]   # <start> ~ 마지막 직전
        trg_label = trg_ids[1:][:self.max_len]    # 두 번째 ~ <end>

        # 고정 길이 패딩
        src_ids   = src_ids   + [PAD_ID] * (self.max_len - len(src_ids))
        trg_input = trg_input + [PAD_ID] * (self.max_len - len(trg_input))
        trg_label = trg_label + [PAD_ID] * (self.max_len - len(trg_label))

        return (
            torch.tensor(src_ids,   dtype=torch.long),
            torch.tensor(trg_input, dtype=torch.long),
            torch.tensor(trg_label, dtype=torch.long),
        )

In [19]:
# ── DataLoader 생성 ───────────────────────────────────────────
MAX_LEN    = 42      # <start>/<end> 포함 여유분
BATCH_SIZE = 64

dataset     = TranslationDataset(kor_encoded, eng_encoded, max_len=MAX_LEN)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 배치 형태 확인
for src, trg_in, trg_lb in data_loader:
    print(f"src shape      : {src.shape}")
    print(f"trg_input shape: {trg_in.shape}")
    print(f"trg_label shape: {trg_lb.shape}")
    break

src shape      : torch.Size([64, 42])
trg_input shape: torch.Size([64, 42])
trg_label shape: torch.Size([64, 42])


## Step 4. 모델 설계

In [20]:
# ─────────────────────────────────────────────────────────────
# [Cell 4] Bahdanau Attention 기반 Seq2Seq 모델 설계
# - BahdanauAttention : Additive Attention으로 가중치 계산
# - Encoder           : GRU 기반 한국어 인코더
# - Decoder           : GRU + Context Vector 기반 영어 디코더
# - Seq2SeqAttention  : 학습/추론 통합 모델
# ─────────────────────────────────────────────────────────────

import torch.nn as nn

class BahdanauAttention(nn.Module):
    """
    Bahdanau (Additive) Attention
    score = v · tanh(W1·H + W2·s_{t-1})
    인코더 전체 출력 H와 디코더 이전 hidden을 비교해
    각 인코더 스텝에 대한 attention 가중치를 계산합니다.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)    # 인코더 출력 변환
        self.W2 = nn.Linear(hidden_dim, hidden_dim)    # 디코더 hidden 변환
        self.v  = nn.Linear(hidden_dim, 1, bias=False) # 스칼라 점수 산출

    def forward(self, hidden, encoder_outputs):
        """
        hidden         : (batch, hidden_dim)
        encoder_outputs: (src_len, batch, hidden_dim)
        반환: attention weights (batch, src_len)
        """
        src_len         = encoder_outputs.shape[0]
        hidden          = hidden.unsqueeze(1).repeat(1, src_len, 1) # (batch, src_len, hidden)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)          # (batch, src_len, hidden)

        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch, src_len, hidden)
        score  = self.v(energy).squeeze(2)                               # (batch, src_len)
        return torch.softmax(score, dim=1)


class Encoder(nn.Module):
    """
    GRU 기반 한국어 인코더
    임베딩 후 GRU로 전체 시퀀스를 인코딩합니다.
    Dropout으로 과적합을 방지합니다.
    """
    def __init__(self, input_dim: int, emb_dim: int, hidden_dim: int, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_ID)
        self.rnn       = nn.GRU(emb_dim, hidden_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        """
        src: (src_len, batch)
        반환: outputs (src_len, batch, hidden_dim), hidden (1, batch, hidden_dim)
        """
        embedded        = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden


class Decoder(nn.Module):
    """
    GRU 기반 영어 디코더
    매 스텝마다 Attention으로 Context Vector를 계산하고
    현재 hidden state와 결합해 다음 토큰을 예측합니다.
    """
    def __init__(self, output_dim: int, emb_dim: int, hidden_dim: int,
                 attention: BahdanauAttention, dropout: float = 0.3):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_ID)
        self.rnn       = nn.GRU(emb_dim, hidden_dim)
        self.fc_out    = nn.Linear(hidden_dim * 2, output_dim)  # hidden + context → vocab
        self.dropout   = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs):
        """
        input          : (batch,)
        hidden         : (1, batch, hidden_dim)
        encoder_outputs: (src_len, batch, hidden_dim)
        반환: prediction (batch, output_dim), hidden, attention weights
        """
        input    = input.unsqueeze(0)                           # (1, batch)
        embedded = self.dropout(self.embedding(input))          # (1, batch, emb_dim)

        # Attention 가중치 → Context Vector 계산
        a          = self.attention(hidden[-1], encoder_outputs) # (batch, src_len)
        a_expanded = a.unsqueeze(1)                              # (batch, 1, src_len)
        enc_perm   = encoder_outputs.permute(1, 0, 2)           # (batch, src_len, hidden)
        context    = torch.bmm(a_expanded, enc_perm)            # (batch, 1, hidden)
        context    = context.permute(1, 0, 2)                   # (1, batch, hidden)

        output, hidden = self.rnn(embedded, hidden)             # (1, batch, hidden)

        # hidden + context → 다음 토큰 예측
        output     = output.squeeze(0)                          # (batch, hidden)
        context    = context.squeeze(0)                         # (batch, hidden)
        prediction = self.fc_out(torch.cat([output, context], dim=1))  # (batch, vocab)
        return prediction, hidden, a


class Seq2SeqAttention(nn.Module):
    """
    Encoder-Decoder 통합 모델
    - 학습 모드 (trg 제공): Teacher Forcing 디코딩
    - 추론 모드 (trg=None): BOS → EOS 자기회귀 디코딩
    """
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg=None, max_len: int = 50):
        batch_size = src.shape[1]
        outputs, attentions = [], []

        encoder_outputs, hidden = self.encoder(src)

        if trg is not None:
            # Teacher Forcing: 정답 토큰을 디코더 입력으로 직접 제공
            for t in range(trg.shape[0]):
                pred, hidden, attn = self.decoder(trg[t], hidden, encoder_outputs)
                outputs.append(pred.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))
        else:
            # 추론: BOS로 시작, 배치 내 모든 문장이 EOS를 생성하면 종료
            input    = torch.full((batch_size,), BOS_ID, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)
            for _ in range(max_len):
                pred, hidden, attn = self.decoder(input, hidden, encoder_outputs)
                outputs.append(pred.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))
                top1      = pred.argmax(1)
                input     = top1
                finished |= (top1 == EOS_ID)
                if finished.all():
                    break

        outputs    = torch.cat(outputs,    dim=0)  # (trg_len, batch, vocab)
        attentions = torch.cat(attentions, dim=0)  # (trg_len, batch, src_len)
        return outputs, attentions


# ── 모델 초기화 ───────────────────────────────────────────────
INPUT_DIM  = len(kor_vocab)
OUTPUT_DIM = len(eng_vocab)
EMB_DIM    = 256
HID_DIM    = 512
DROPOUT    = 0.3

encoder   = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, DROPOUT).to(device)
attention = BahdanauAttention(HID_DIM).to(device)
decoder   = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, attention, DROPOUT).to(device)
model     = Seq2SeqAttention(encoder, decoder, device).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 가능한 파라미터 수: {total_params:,}")

학습 가능한 파라미터 수: 42,856,172


## Step 5. 모델 훈련

In [ ]:
# ─────────────────────────────────────────────────────────────
# [Cell 5] 모델 학습 + 추론 + Attention Map 시각화
# - train_step(): Teacher Forcing 방식 학습
# - translate() : OKT 형태소 분석 → 추론
# - 매 에포크마다 지정 예문 4개 번역 결과 출력
# - 학습 완료 후 최고 번역 케이스 Attention Map 시각화
# ─────────────────────────────────────────────────────────────

import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# ── 지정 예문 4개 및 실제 정답 번역 ──────────────────────────
sample_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다.",
]

reference_translations = [
    "obama is the president .",
    "citizens live in the city .",
    "coffee is not necessary .",
    "seven people were killed .",
]


# ── 번역 함수 ─────────────────────────────────────────────────
def translate(sentence: str, model, kor_vocab: dict, id2eng: dict,
              okt, max_len: int = 50) -> tuple:
    """
    단일 한국어 문장 → 영어 번역
    반환: (번역 토큰 리스트, attention 행렬, 소스 형태소 리스트)
    """
    model.eval()
    with torch.no_grad():
        clean   = preprocess_korean(sentence)
        tokens  = okt.morphs(clean, norm=True, stem=True)
        src_ids = [kor_vocab.get(t, UNK_ID) for t in tokens]
        src     = torch.tensor(src_ids, dtype=torch.long).unsqueeze(1).to(device)

        outputs, attentions = model(src, trg=None, max_len=max_len)

        pred_ids   = outputs.argmax(2).squeeze(1).tolist()
        translated = []
        for pid in pred_ids:
            word = id2eng.get(pid, "<unk>")
            if word == "<end>":
                break
            if word not in ("<start>", "<pad>"):
                translated.append(word)

        # (trg_len, 1, src_len) → (trg_len, src_len)
        attn = attentions.squeeze(1).cpu().numpy()
        return translated, attn, tokens


# ── Attention Map 시각화 함수 ─────────────────────────────────
def plot_attention(src_tokens: list, trg_tokens: list,
                   attention: np.ndarray, title: str = "Attention Map"):
    """
    src_tokens: 한국어 형태소 리스트 (x축)
    trg_tokens: 생성된 영어 토큰 리스트 (y축)
    attention : (trg_len, src_len) numpy 배열
    """
    attn = attention[:len(trg_tokens), :len(src_tokens)]

    fig, ax = plt.subplots(
        figsize=(max(8, len(src_tokens) * 0.8),
                 max(5, len(trg_tokens) * 0.6))
    )
    im = ax.imshow(attn, cmap="YlOrRd", aspect="auto")

    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha="right", fontproperties=fontprop)
    ax.set_yticklabels(trg_tokens)
    ax.set_xlabel("Source (Korean)", fontproperties=fontprop)
    ax.set_ylabel("Target (English)", fontproperties=fontprop)
    ax.set_title(title, fontproperties=fontprop)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


# ── 학습 함수 ─────────────────────────────────────────────────
def train_step(model, data_loader, optimizer, criterion, epoch: int) -> float:
    """
    한 에포크 학습
    - Teacher Forcing: 디코더 입력으로 정답 토큰을 직접 제공
    - Gradient Clipping: 기울기 폭주 방지
    """
    model.train()
    epoch_loss = 0
    bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)

    for src, trg_input, trg_label in bar:
        src       = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)

        optimizer.zero_grad()
        outputs, _ = model(src, trg_input)

        loss = criterion(
            outputs.reshape(-1, outputs.shape[-1]),
            trg_label.reshape(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

    return epoch_loss / len(data_loader)


# ── 번역 결과 출력 및 누적 저장 함수 ─────────────────────────
def print_translations(epoch: int, best_results: dict):
    """
    예문 4개 번역 결과를 출력하고 에포크별로 누적 저장합니다.
    """
    print(f"\n{'='*60}")
    print(f"  Epoch {epoch+1} 번역 결과")
    print(f"{'='*60}")

    for i, (sent, ref) in enumerate(zip(sample_sentences, reference_translations)):
        result, attn, src_toks = translate(sent, model, kor_vocab, id2eng, okt)
        translation = " ".join(result) if result else "(생성 실패)"

        print(f"\n  K{i+1}) 입력: {sent}")
        print(f"       정답: {ref}")
        print(f"       번역: {translation}")

        if i not in best_results:
            best_results[i] = []
        best_results[i].append({
            "epoch"      : epoch + 1,
            "translation": translation,
            "attn"       : attn,
            "src_toks"   : src_toks,
            "sentence"   : sent,
            "reference"  : ref,
        })

    print(f"\n{'='*60}\n")
    return best_results


# ── 훈련 실행 ─────────────────────────────────────────────────
EPOCHS = 12
best_results = {}

for epoch in range(EPOCHS):
    train_loss = train_step(model, data_loader, optimizer, criterion, epoch)
    print(f"\n[Epoch {epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f}")
    best_results = print_translations(epoch, best_results)


# ── 최고 번역 케이스 선정 및 Attention Map 시각화 ─────────────
# 마지막 5개 에포크 중 정답과 토큰 수가 가장 근접한 결과를 최고 케이스로 선정
print("\n" + "="*60)
print("  최고 번역 케이스 & Attention Map 시각화")
print("="*60)

for i in range(len(sample_sentences)):
    candidates = best_results[i]
    last5      = candidates[-5:]

    # 정답 토큰 수와 번역 토큰 수의 차이가 가장 작은 것을 최고 케이스로 선정
    ref_len = len(reference_translations[i].split())
    best    = min(
        last5,
        key=lambda x: abs(len(x["translation"].split()) - ref_len)
                      if x["translation"] != "(생성 실패)" else float("inf")
    )

    print(f"\nK{i+1}) 입력  : {best['sentence']}")
    print(f"     정답  : {best['reference']}")
    print(f"     번역  : {best['translation']}")
    print(f"     Epoch : {best['epoch']}")

    if best["translation"] != "(생성 실패)":
        plot_attention(
            src_tokens=best["src_toks"],
            trg_tokens=best["translation"].split(),
            attention=best["attn"],
            title=f"K{i+1}) {best['sentence']} (Epoch {best['epoch']})"
        )

Epoch 1: 100%|██████████| 1084/1084 [07:48<00:00,  2.31it/s, loss=5.0451]



[Epoch 1/12] Train Loss: 5.4966

  Epoch 1 번역 결과

  K1) 입력: 오바마는 대통령이다.
       정답: obama is the president .
       번역: obama s president elect barack obama is obama to president barack obama s inauguration .

  K2) 입력: 시민들은 도시 속에 산다.
       정답: citizens live in the city .
       번역: the torch relay to be held in the area .

  K3) 입력: 커피는 필요 없다.
       정답: coffee is not necessary .
       번역: no matter was not yet to be no matter .

  K4) 입력: 일곱 명의 사망자가 발생했다.
       정답: seven people were killed .
       번역: the deaths occurred in the blast , killing people died in the southern city of kandahar , killing people .




Epoch 2: 100%|██████████| 1084/1084 [07:51<00:00,  2.30it/s, loss=4.6522]



[Epoch 2/12] Train Loss: 4.4250

  Epoch 2 번역 결과

  K1) 입력: 오바마는 대통령이다.
       정답: obama is the president .
       번역: obama is the president elect barack obama s transition to president barack obama s inauguration .

  K2) 입력: 시민들은 도시 속에 산다.
       정답: citizens live in the city .
       번역: cities officials say they want to cities cities cities cities cities cities cities cities cities cities cities cities cities cities cities cities cities .

  K3) 입력: 커피는 필요 없다.
       정답: coffee is not necessary .
       번역: but he insisted that the couple was not involved in the room but no one knows that there was no reason to be able to be able to do not want to be no .

  K4) 입력: 일곱 명의 사망자가 발생했다.
       정답: seven people were killed .
       번역: deaths died in a death toll , killing at least people dead dead , injuring death toll , a day after the death toll killed at least people dead .




Epoch 3:  47%|████▋     | 514/1084 [03:44<04:09,  2.28it/s, loss=3.9188]

okt가 아닌 mecab을 사용하면 형태소 분석이 더 짧게 걸리지만, 주피터 노트북에서는 mecab이 잘 호환되지 않아 okt를 사용했는데 시간이 엄청 오래 걸렸다. kiwi를 사용한 다른 그루분도 있었는데 kiwi가 더 빠르게 실행된다.  epoch 20번은 너무 오래 걸려서 12번에서 멈추고 epoch은 10번만 하는 걸로 수정하였다. learning rate을 좀 줄이고, 번역 결과 같은 단어(premium, obama city 등)가 반복되어 teacher forcing ratio를 적용하여 다시 epoch 10회 돌리다가 train loss도 크고 같은 단어의 반복이 너무 많아 중간에 중단하였다.

![](./6.png)

회고  
처음 코드 12번째 epoch에서 문장 4가 그나마 제일 잘 번역된 것 같다. 다른 그루 분들의 코드를 참고하니 데이터 전처리 과정에서 데이터의 개수가 많이 줄어들어 훈련 시간이 적었고 loss가 적다고 번역이 잘된 것은 아닌 것 같다(과적합). 처음 코드의 번역결과가 두번째 코드 보다 더 좋은데 앞으로는 첫번째 코드를 그대로 두고 수정한 코드를 밑에 추가로 작성하는 방식으로 수정을 하여 두 코드의 결과를 동시에 비교할 수 있도록 수정방법을 보완해야겠다. 데이터 정제과정에서 앞에 3만개만 실행하였더니 번역능력이 처음 코드보다 떨어졌다. 